In [1]:
from __future__ import division, print_function
import sys, os, glob, time, warnings, gc
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table, vstack, hstack
import fitsio
# from astropy.io import fits

In [2]:
params = {'legend.fontsize': 'large',
         'axes.labelsize': 'large',
         'axes.titlesize':'large',
         'xtick.labelsize':'large',
         'ytick.labelsize':'large',
         'figure.facecolor':'w'} 
plt.rcParams.update(params)

In [3]:
n_exp = 1
overwrite = False

# indir = os.getenv('INDIR')
# output_dir = os.getenv('OUTDIR')
indir = '/global/cfs/cdirs/desi/spectro/redux/cascades/exposures'
os.environ['INDIR'] = indir


# David Schlegel's list of exposures with speed == r_depth / exptime > 0.3
explist_dark = Table.read('/global/cfs/cdirs/desi/users/rongpu/spectro/sv1/explist-deep-dark.txt', format='ascii')
explist_bright = Table.read('/global/cfs/cdirs/desi/users/rongpu/spectro/sv1/explist-deep-bright.txt', format='ascii')

explist = vstack([explist_dark, explist_bright], join_type='exact')
print(len(explist))

96


In [4]:
explist

TILEID,NIGHT,EXPID
int64,int64,int64
80607,20201214,67744
80607,20201214,67765
80607,20201214,67766
80607,20201214,67767
80607,20201214,67768
80608,20201214,67769
80608,20201214,67770
80608,20201214,67771
80609,20201214,67781


In [5]:
cframes_stack = []

for index in range(len(explist)):
    tileid = explist['TILEID'][index]
    night = explist['NIGHT'][index]
    expid = explist['EXPID'][index]
    exposure_dir = os.path.join(indir, str(night), str(expid).zfill(8))
    cframe_list = glob.glob(os.path.join(exposure_dir, 'cframe-*'))
    if len(cframe_list)==0:
        print(exposure_dir, 'is empty')
        continue

    cframes = Table()
    cframes['cframe_fn'] = np.array(cframe_list)
    cframes['tileid'] = np.zeros(len(cframes), dtype=int)
    cframes['expid'] = np.zeros(len(cframes), dtype=int)
    cframes['camera'] = ' '
    cframes['petal_loc'] = -1 * np.ones(len(cframes), dtype=np.int32)

    for cframe_index, cframe_fn in enumerate(cframe_list):
        _, cp, expid1 = os.path.basename(cframe_fn).strip('.fits').split('-')
        camera, petal_loc = cp[0], cp[1]
        expid1 = int(expid)
        if expid1!=expid:
            raise ValueError()
        cframes['tileid'][cframe_index] = tileid
        cframes['expid'][cframe_index] = expid
        cframes['camera'][cframe_index] = camera
        cframes['petal_loc'][cframe_index] = petal_loc
    
    cframes_stack.append(cframes)
    
cframes = vstack(cframes_stack)
print(len(cframes))
print(len(np.unique(cframes['expid'])))

cframes[:1]

2835
96


cframe_fn,tileid,expid,camera,petal_loc
str97,int64,int64,str1,int32
/global/cfs/cdirs/desi/spectro/redux/cascades/exposures/20201214/00067744/cframe-r1-00067744.fits,80607,67744,r,1


In [6]:
cframes['keep'] = False
for expid in np.unique(cframes['expid']):
    mask = cframes['expid']==expid
    for petal_loc in np.arange(10):
        mask1 = mask & (cframes['petal_loc']==petal_loc)
        if np.sum(mask1)==3:  # require all 3 cameras
            cframes['keep'][mask1] = True
print(np.sum(cframes['keep']), np.sum(~cframes['keep']))

cframes = cframes[cframes['keep']]
print(len(cframes))

2835 0
2835


In [7]:
# mask = cframes['tileid']==65008
# cframes = cframes[mask]

In [8]:
# keep only one camera for simplicity
mask = cframes['camera']=='b'
cframes = cframes[mask]
print(len(cframes))

cframes.sort('cframe_fn')

945


------
## 4X depth coadds

Initial setup:
```
source /global/cfs/cdirs/desi/software/desi_environment.sh 21.2
module load parallel

export INDIR=/global/cfs/cdirs/desi/spectro/redux/cascades/exposures
export OUTDIR=/global/cfs/cdirs/desi/users/$USER/redux/cascades/4x_depth
```

Run desi_group_spectra:
```
salloc -N 1 -C haswell -q interactive -t 4:00:00
parallel --jobs 32 < desi_group_spectra_4x.txt ; exit
```

Run desi_coadd_spectra:
```
salloc -N 1 -C haswell -q interactive -t 4:00:00
parallel --jobs 32 < desi_coadd_spectra_4x.txt ; exit
```

Run redrock:
```
salloc -N 20 --qos interactive -C haswell -t 4:00:00
parallel --jobs 20 --delay 1 < rrdesi_4x.txt ; exit
```

In [9]:
# output_dir = os.getenv('OUTDIR')
outdir = os.path.expandvars('/global/cfs/cdirs/desi/users/$USER/redux/cascades/4x_depth')
os.environ['OUTDIR'] = outdir

__List of exposures for each subset__

In [10]:
# # The original run that includes exposures affected by the "fiducials on" issue
# subsamp_dict = \
# {'80605': [[73702, 74779, 68291, 74780, 68292, 74782, 67972]],
#  '80606': [[67970, 68813, 68285, 68289, 68630, 67968, 67971]],
#  '80607': [[68847, 68845, 68028, 68664, 69440, 67767, 67768],
#   [67744, 68662, 68666, 68846, 69441, 68844]],
#  '80608': [[68327, 68660, 69249, 67770, 68317],
#   [68024, 68661, 68842, 68328, 67769, 68841],
#   [69252, 68025, 69438, 68026, 67771, 69251]],
#  '80609': [[68490, 68339, 68337, 68334, 68065, 68336],
#   [67781, 68063, 68064, 67783, 68340, 68489, 68338]],
#  '80610': [[68332, 68478, 68041, 68330, 68488, 68683, 75114],
#   [75113, 68477, 68040, 68333, 68331, 68042, 75116]],
#  '80613': [[69228, 68659],
#   [68658, 69229],
#   [68657],
#   [69227, 69226, 69230, 69225]]}

In [11]:
# # Full list
# subsamp_dict = \
# {'80605': [[73702, 74779, 68291, 74780, 68292, 74782, 67972]],
#  '80606': [[67970, 68813, 68285, 68289, 68630, 67968, 67971]],
#  '80607': [[67767, 68028, 68847, 68664, 68845, 68846],
#   [67765, 68662, 68663, 68027, 67766, 68666]],
#  '80608': [[68842, 69252, 68023, 68024, 68661],
#   [69253, 68491, 67769, 69249, 67770, 68328, 67771],
#   [68660, 68327, 68317, 68025, 68841]],
#  '80609': [[68490, 68339, 68337, 68334, 68065, 68336],
#   [67781, 68063, 68064, 67783, 68340, 68489, 68338]],
#  '80610': [[68332, 68478, 68041, 68330, 68488, 68683, 75114],
#   [75113, 68477, 68040, 68333, 68331, 68042, 75116]],
#  '80613': [[69228, 68659],
#   [68658, 69229],
#   [68657],
#   [69227, 69226, 69230, 69225]]}

In [12]:
# Only the subsets for the rerun
subsamp_dict = \
{'80607': [[67767, 68028, 68847, 68664, 68845, 68846],
  [67765, 68662, 68663, 68027, 67766, 68666]],
 '80608': [[68842, 69252, 68023, 68024, 68661],
  [69253, 68491, 67769, 69249, 67770, 68328, 67771],
  [68660, 68327, 68317, 68025, 68841]]}

__Print desi_group_spectra commands__

In [13]:
command_output_path_all = 'desi_group_spectra_4x_all.txt'
command_output_path = 'desi_group_spectra_4x.txt'  # commands for reruns (e.g. after failures)

f_all = open(command_output_path_all, 'w')
f = open(command_output_path, 'w')

for tileid_str in subsamp_dict.keys():
    
    # tileid_str = '80605'
    tileid = int(tileid_str)

    for subset_index, subset in enumerate(subsamp_dict[tileid_str]):
        # print(subset)

        mask = np.in1d(cframes['expid'], subset)
        petal_locs = np.unique(cframes['petal_loc'][mask])

        for petal_loc in petal_locs:

            input_locs = []
            for expid in subset:
                mask = (cframes['expid']==expid) & (cframes['petal_loc']==petal_loc)
                if np.sum(mask)>1:
                    raise ValueError
                if np.sum(mask)==1:
                    exposure_dir = os.path.dirname(cframes['cframe_fn'][mask][0])
                    input_locs.append(os.path.join(exposure_dir.replace(indir, '$INDIR'), 'cframe-[brz]{}-*.fits'.format(petal_loc)))

            input_locs = ' '.join(input_locs)
            spectra_output_loc = os.path.join('$OUTDIR', str(tileid), 'spectra-{}-{}-subset-{}.fits'.format(petal_loc, tileid, subset_index+1))
            
            f_all.write('desi_group_spectra --inframes {} --outfile {}\n'.format(input_locs, spectra_output_loc))

            if os.path.isfile(os.path.expandvars(spectra_output_loc)) and (not overwrite):
                # print('Warninig: {} already exists!'.format(spectra_output_loc))
                pass
            else:
                f.write('desi_group_spectra --inframes {} --outfile {}\n'.format(input_locs, spectra_output_loc))
                
f_all.close()
f.close()

__Print desi_coadd_spectra commands__

In [14]:
command_output_path_all = 'desi_coadd_spectra_4x_all.txt'
command_output_path = 'desi_coadd_spectra_4x.txt'  # commands for reruns (e.g. after failures)

f_all = open(command_output_path_all, 'w')
f = open(command_output_path, 'w')

for tileid_str in subsamp_dict.keys():
    
    # tileid_str = '80605'
    tileid = int(tileid_str)

    for subset_index, subset in enumerate(subsamp_dict[tileid_str]):
        # print(subset)

        mask = np.in1d(cframes['expid'], subset)
        petal_locs = np.unique(cframes['petal_loc'][mask])

        for petal_loc in petal_locs:

            input_locs = []
            for expid in subset:
                mask = (cframes['expid']==expid) & (cframes['petal_loc']==petal_loc)
                if np.sum(mask)>1:
                    raise ValueError
                if np.sum(mask)==1:
                    exposure_dir = os.path.dirname(cframes['cframe_fn'][mask][0])
                    input_locs.append(os.path.join(exposure_dir.replace(indir, '$INDIR'), 'cframe-[brz]{}-*.fits'.format(petal_loc)))

            input_locs = ' '.join(input_locs)
            coadd_output_loc = os.path.join('$OUTDIR', str(tileid), 'coadd-{}-{}-subset-{}.fits'.format(petal_loc, tileid, subset_index+1))
            
            f_all.write('desi_coadd_spectra -i {} -o {}\n'.format(input_locs, coadd_output_loc))

            if os.path.isfile(os.path.expandvars(coadd_output_loc)) and (not overwrite):
                # print('Warninig: {} already exists!'.format(coadd_output_loc))
                pass
            else:
                f.write('desi_coadd_spectra -i {} -o {}\n'.format(input_locs, coadd_output_loc))

f_all.close()
f.close()

__Print redrock commands__  
Include 10-minute timeout

In [15]:
command_output_path_all = 'rrdesi_4x_all.txt'
command_output_path = 'rrdesi_4x.txt'  # commands for reruns (e.g. after failures)

f_all = open(command_output_path_all, 'w')
f = open(command_output_path, 'w')

for tileid_str in subsamp_dict.keys():

    tileid = int(tileid_str)
    
    for subset_index, subset in enumerate(subsamp_dict[tileid_str]):

        mask = np.in1d(cframes['expid'], subset)
        petal_locs = np.unique(cframes['petal_loc'][mask])

        for petal_loc in petal_locs:
        
            spectra_output_loc = os.path.join('$OUTDIR', str(tileid), 'spectra-{}-{}-subset-{}.fits'.format(petal_loc, tileid, subset_index+1))
            redrock_output_loc = os.path.join('$OUTDIR', str(tileid), 'redrock-{}-{}-subset-{}.h5'.format(petal_loc, tileid, subset_index+1))
            zbest_output_loc = os.path.join('$OUTDIR', str(tileid), 'zbest-{}-{}-subset-{}.fits'.format(petal_loc, tileid, subset_index+1))
            log_loc = os.path.join('$OUTDIR', str(tileid), 'redrock-{}-{}-subset-{}.log'.format(petal_loc, tileid, subset_index+1))

            f_all.write('srun -N 1 -n 32 -c 2 -t 00:10:00 rrdesi_mpi {} -o {} -z {} &> {}\n'.format(spectra_output_loc, redrock_output_loc, zbest_output_loc, log_loc))
        
            if os.path.isfile(os.path.expandvars(zbest_output_loc)) and (not overwrite):
                # print('Warninig: {} already exists!'.format(zbest_output_loc))
                pass
            else:
                f.write('srun -N 1 -n 32 -c 2 -t 00:10:00 rrdesi_mpi {} -o {} -z {} &> {}\n'.format(spectra_output_loc, redrock_output_loc, zbest_output_loc, log_loc))
f.close()
f_all.close()

------
## 3X depth coadds

Initial setup:
```
source /global/cfs/cdirs/desi/software/desi_environment.sh 21.2
module load parallel

export INDIR=/global/cfs/cdirs/desi/spectro/redux/cascades/exposures
export OUTDIR=/global/cfs/cdirs/desi/users/$USER/redux/cascades/3x_depth
```

Run desi_group_spectra:
```
salloc -N 1 -C haswell -q interactive -t 4:00:00
parallel --jobs 32 < desi_group_spectra_3x.txt ; exit
```

Run desi_coadd_spectra:
```
salloc -N 1 -C haswell -q interactive -t 4:00:00
parallel --jobs 32 < desi_coadd_spectra_3x.txt ; exit
```

Run redrock:
```
salloc -N 5 --qos interactive -C haswell -t 4:00:00
parallel --jobs 5 --delay 1 < rrdesi_3x.txt ; exit
```

In [14]:
# output_dir = os.getenv('OUTDIR')
outdir = os.path.expandvars('/global/cfs/cdirs/desi/users/$USER/redux/cascades/3x_depth')
os.environ['OUTDIR'] = outdir

__List of exposures for each subset__

In [15]:
subsamp_dict = \
{'80605': [[74779, 73702, 68290, 74781, 74783, 67975],
  [67972, 74782, 68292, 74780, 68291]],
 '80606': [[68285, 68284, 67971, 68630], [68288, 68812, 68813, 67970, 68289]]}

__Print desi_group_spectra commands__

In [16]:
command_output_path_all = 'desi_group_spectra_3x_all.txt'
command_output_path = 'desi_group_spectra_3x.txt'  # commands for reruns (e.g. after failures)

f_all = open(command_output_path_all, 'w')
f = open(command_output_path, 'w')

for tileid_str in subsamp_dict.keys():
    
    # tileid_str = '80605'
    tileid = int(tileid_str)

    for subset_index, subset in enumerate(subsamp_dict[tileid_str]):
        # print(subset)

        mask = np.in1d(cframes['expid'], subset)
        petal_locs = np.unique(cframes['petal_loc'][mask])

        for petal_loc in petal_locs:

            input_locs = []
            for expid in subset:
                mask = (cframes['expid']==expid) & (cframes['petal_loc']==petal_loc)
                if np.sum(mask)>1:
                    raise ValueError
                if np.sum(mask)==1:
                    exposure_dir = os.path.dirname(cframes['cframe_fn'][mask][0])
                    input_locs.append(os.path.join(exposure_dir.replace(indir, '$INDIR'), 'cframe-[brz]{}-*.fits'.format(petal_loc)))

            input_locs = ' '.join(input_locs)
            spectra_output_loc = os.path.join('$OUTDIR', str(tileid), 'spectra-{}-{}-subset-{}.fits'.format(petal_loc, tileid, subset_index+1))
            
            f_all.write('desi_group_spectra --inframes {} --outfile {}\n'.format(input_locs, spectra_output_loc))

            if os.path.isfile(os.path.expandvars(spectra_output_loc)) and (not overwrite):
                print('Warninig: {} already exists!'.format(spectra_output_loc))
            else:
                f.write('desi_group_spectra --inframes {} --outfile {}\n'.format(input_locs, spectra_output_loc))

f_all.close()
f.close()

Warninig: $OUTDIR/80605/spectra-0-80605-subset-1.fits already exists!
Warninig: $OUTDIR/80605/spectra-1-80605-subset-1.fits already exists!
Warninig: $OUTDIR/80605/spectra-2-80605-subset-1.fits already exists!
Warninig: $OUTDIR/80605/spectra-3-80605-subset-1.fits already exists!
Warninig: $OUTDIR/80605/spectra-4-80605-subset-1.fits already exists!
Warninig: $OUTDIR/80605/spectra-5-80605-subset-1.fits already exists!
Warninig: $OUTDIR/80605/spectra-6-80605-subset-1.fits already exists!
Warninig: $OUTDIR/80605/spectra-7-80605-subset-1.fits already exists!
Warninig: $OUTDIR/80605/spectra-8-80605-subset-1.fits already exists!
Warninig: $OUTDIR/80605/spectra-9-80605-subset-1.fits already exists!
Warninig: $OUTDIR/80605/spectra-0-80605-subset-2.fits already exists!
Warninig: $OUTDIR/80605/spectra-1-80605-subset-2.fits already exists!
Warninig: $OUTDIR/80605/spectra-2-80605-subset-2.fits already exists!
Warninig: $OUTDIR/80605/spectra-3-80605-subset-2.fits already exists!
Warninig: $OUTDIR/80

__Print desi_coadd_spectra commands__

In [17]:
command_output_path_all = 'desi_coadd_spectra_3x_all.txt'
command_output_path = 'desi_coadd_spectra_3x.txt'  # commands for reruns (e.g. after failures)

f_all = open(command_output_path_all, 'w')
f = open(command_output_path, 'w')

for tileid_str in subsamp_dict.keys():
    
    # tileid_str = '80605'
    tileid = int(tileid_str)

    for subset_index, subset in enumerate(subsamp_dict[tileid_str]):
        # print(subset)

        mask = np.in1d(cframes['expid'], subset)
        petal_locs = np.unique(cframes['petal_loc'][mask])

        for petal_loc in petal_locs:

            input_locs = []
            for expid in subset:
                mask = (cframes['expid']==expid) & (cframes['petal_loc']==petal_loc)
                if np.sum(mask)>1:
                    raise ValueError
                if np.sum(mask)==1:
                    exposure_dir = os.path.dirname(cframes['cframe_fn'][mask][0])
                    input_locs.append(os.path.join(exposure_dir.replace(indir, '$INDIR'), 'cframe-[brz]{}-*.fits'.format(petal_loc)))

            input_locs = ' '.join(input_locs)
            coadd_output_loc = os.path.join('$OUTDIR', str(tileid), 'coadd-{}-{}-subset-{}.fits'.format(petal_loc, tileid, subset_index+1))
            
            f_all.write('desi_coadd_spectra -i {} -o {}\n'.format(input_locs, coadd_output_loc))

            if os.path.isfile(os.path.expandvars(coadd_output_loc)) and (not overwrite):
                print('Warninig: {} already exists!'.format(coadd_output_loc))
            else:
                f.write('desi_coadd_spectra -i {} -o {}\n'.format(input_locs, coadd_output_loc))

f_all.close()
f.close()

__Print redrock commands__  
Include 10-minute timeout

In [18]:
command_output_path_all = 'rrdesi_3x_all.txt'
command_output_path = 'rrdesi_3x.txt'  # commands for reruns (e.g. after failures)

f_all = open(command_output_path_all, 'w')
f = open(command_output_path, 'w')

for tileid_str in subsamp_dict.keys():

    tileid = int(tileid_str)
    
    for subset_index, subset in enumerate(subsamp_dict[tileid_str]):

        mask = np.in1d(cframes['expid'], subset)
        petal_locs = np.unique(cframes['petal_loc'][mask])

        for petal_loc in petal_locs:
        
            spectra_output_loc = os.path.join('$OUTDIR', str(tileid), 'spectra-{}-{}-subset-{}.fits'.format(petal_loc, tileid, subset_index+1))
            redrock_output_loc = os.path.join('$OUTDIR', str(tileid), 'redrock-{}-{}-subset-{}.h5'.format(petal_loc, tileid, subset_index+1))
            zbest_output_loc = os.path.join('$OUTDIR', str(tileid), 'zbest-{}-{}-subset-{}.fits'.format(petal_loc, tileid, subset_index+1))
            log_loc = os.path.join('$OUTDIR', str(tileid), 'redrock-{}-{}-subset-{}.log'.format(petal_loc, tileid, subset_index+1))

            f_all.write('srun -N 1 -n 32 -c 2 -t 00:10:00 rrdesi_mpi {} -o {} -z {} &> {}\n'.format(spectra_output_loc, redrock_output_loc, zbest_output_loc, log_loc))
        
            if os.path.isfile(os.path.expandvars(zbest_output_loc)) and (not overwrite):
                print('Warninig: {} already exists!'.format(zbest_output_loc))
            else:
                f.write('srun -N 1 -n 32 -c 2 -t 00:10:00 rrdesi_mpi {} -o {} -z {} &> {}\n'.format(spectra_output_loc, redrock_output_loc, zbest_output_loc, log_loc))
f.close()
f_all.close()

Warninig: $OUTDIR/80605/zbest-0-80605-subset-1.fits already exists!
Warninig: $OUTDIR/80605/zbest-1-80605-subset-1.fits already exists!
Warninig: $OUTDIR/80605/zbest-2-80605-subset-1.fits already exists!
Warninig: $OUTDIR/80605/zbest-3-80605-subset-1.fits already exists!
Warninig: $OUTDIR/80605/zbest-4-80605-subset-1.fits already exists!
Warninig: $OUTDIR/80605/zbest-5-80605-subset-1.fits already exists!
Warninig: $OUTDIR/80605/zbest-6-80605-subset-1.fits already exists!
Warninig: $OUTDIR/80605/zbest-7-80605-subset-1.fits already exists!
Warninig: $OUTDIR/80605/zbest-8-80605-subset-1.fits already exists!
Warninig: $OUTDIR/80605/zbest-9-80605-subset-1.fits already exists!
Warninig: $OUTDIR/80605/zbest-0-80605-subset-2.fits already exists!
Warninig: $OUTDIR/80605/zbest-1-80605-subset-2.fits already exists!
Warninig: $OUTDIR/80605/zbest-2-80605-subset-2.fits already exists!
Warninig: $OUTDIR/80605/zbest-3-80605-subset-2.fits already exists!
Warninig: $OUTDIR/80605/zbest-4-80605-subset-2.f